<h1>Chapter 10 - Creating Text Embedding Models</h1>
<i>Exploring methods for both training and fine-tuning embedding models.</i>

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter10/Chapter%2010%20-%20Creating%20Text%20Embedding%20Models.ipynb)

---

This notebook is for Chapter 10 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [ ]:
# %%capture
# !pip install -q accelerate>=0.27.2 peft>=0.9.0 bitsandbytes>=0.43.0 transformers>=4.38.2 trl>=0.7.11 sentencepiece>=0.1.99
# !pip install -q sentence-transformers>=3.0.0 mteb>=1.1.2 datasets>=2.18.0

# Creating an Embedding Model

## **Data（加载对比数据集）**

In [ ]:
from datasets import load_dataset

"""
GLUE（General Language Understanding Evaluation）通用语言理解评估基准
GLUE 基准的任务之一是多类型自然语言推理（MNLI）语料库，它包含 392 702 个有推理关系标注（矛盾、中性、蕴含）的句子对。
核心目的：摆脱过去一个模型只能做单一任务（如仅分类或仅翻译）的局限，评估模型是否具备“通用”的语言理解能力。

MNLI（Multi-Genre Natural Language Inference）多领域自然语言推理
核心目的：测试模型能否理解两句话之间的逻辑因果关系
"""

# Load MNLI dataset from GLUE
# 0 = entailment, 1 = neutral, 2 = contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))  # split=train
train_dataset = train_dataset.remove_columns("idx")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:

"""
这展示了一个前提和假设之间存在蕴含关系的示例，因为它们是正相关的

{'premise': 'One of our number will carry out your instructions minutely.',                 # 前提
 'hypothesis': 'A member of my team will execute your orders with immense precision.',      # 假设
 'label': 0}                                                                                # 标签/标准答案， 0 = 蕴含，1 = 中性，2 = 矛盾
"""
train_dataset[2]

{'premise': 'One of our number will carry out your instructions minutely.',
 'hypothesis': 'A member of my team will execute your orders with immense precision.',
 'label': 0}

## **Model（加载基座嵌入模型）**

In [ ]:
from sentence_transformers import SentenceTransformer

# 我们将使用 BERT 基座模型（不区分大小写版）作为入门模型
# Use a base model
embedding_model = SentenceTransformer('bert-base-uncased')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## **Loss Function（默认损失函数）**

In [ ]:
from sentence_transformers import losses

"""
我们使用 softmax 损失函数来训练模型，为的是说明最早的 sentence-transformers 模型是如何训练的。
实际上，可供选择的损失函数种类繁多，我们通常不建议使用 softmax，因为其他损失函数可能更高效
"""
# Define the loss function. In soft-max loss, we will also need to explicitly set the number of labels.
train_loss = losses.SoftmaxLoss(
    model=embedding_model,
    sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
    num_labels=3  # 失函数你最终分类任务的类别总数，对应的是MNLI的label  0: 蕴含 (Entailment) 1: 中立 (Neutral) 2: 矛盾 (Contradiction)
)

## Evaluation（评估器）

In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

"""
语义文本相似度基准（Semantic Textual Similarity Benchmark，STSB）
我们使用这个数据集来探索模型在语义相似度任务上的表现。此外，我们还要处理 STSB 数据，确保所有的值都在 0 和 1 之间。

"""
# 为STSB创建嵌入相似度评估器
# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')  # split=validation
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score / 5 for score in val_sts["label"]],
    # 在原始的 GLUE STS-B 数据集中，人类给出的相似度评分是 0.0 到 5.0 之间的连续浮点数（5 分代表含义完全相同，0 分代表完全无）
    main_similarity="cosine",  # 指定评估器在比对两组句子的向量时，使用余弦相似度（Cosine Similarity）
)

## **Training（训练）**

In [ ]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# 定义训练参数
# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="base_embedding_model",
    num_train_epochs=1,  #训练轮次。为了加快训练速度，我们将其设为 1，但通常建议把这个值设置得大一些
    per_device_train_batch_size=32,  # 在训练过程中每个设备（如 GPU 或 CPU）同时处理的样本数量。一般来说，该参数的值越大，训练速度越快
    per_device_eval_batch_size=32,  #在评估过程中每个设备（如 GPU 或 CPU）同时处理的样本数量。一般来说，该参数的值越大，评估速度越快
    warmup_steps=100,  # 学习率从 0 线性增加到初始学习率所需的步数。注意，在本次训练过程中，我们没有指定自定义的学习率
    fp16=True,  # 启用此参数后，我们可以进行混合精度训练，使用 16 位浮点数（FP16）而不是默认的32 位浮点数（FP32）进行计算。这可以减少内存使用量并有可能提高训练速度
    eval_steps=100,
    logging_steps=100,
)

In [ ]:
from sentence_transformers.trainer import SentenceTransformerTrainer

# 训练模型
# Train embedding model
trainer = SentenceTransformerTrainer(
    model=embedding_model,  # 嵌入模型
    args=args,  # 训练参数
    train_dataset=train_dataset,  # 训练数据
    loss=train_loss,  # 损失函数
    evaluator=evaluator  # 评估器
)
trainer.train()

Step,Training Loss
100,1.080700
200,0.959400
300,0.916200
400,0.870200
500,0.849100
600,0.854200
700,0.835200
800,0.825200
900,0.818100
1000,0.800300


Computing widget examples:   0%|          | 0/5 [00:00<?, ?example/s]

Computing widget examples:   0%|          | 0/5 [00:00<?, ?example/s]

Computing widget examples:   0%|          | 0/5 [00:00<?, ?example/s]

TrainOutput(global_step=1563, training_loss=0.8453957184872716, metrics={'train_runtime': 372.5713, 'train_samples_per_second': 134.202, 'train_steps_per_second': 4.195, 'total_flos': 0.0, 'train_loss': 0.8453957184872716, 'epoch': 1.0})

In [ ]:

"""
两个核心统计指标：数据通过两种统计学相关系数来评估模型的表现，分值都在 [-1, 1] 之间，越接近 1 代表模型越聪明、越符合人类直觉

（1）Pearson（皮尔逊相关系数）：衡量模型打分与人类打分之间的线性关系（是否呈直线正比）
（2）Spearman（斯皮尔曼等级相关系数）：衡量模型打分与人类打分之间的单调关系（排名先后顺序）。即使数值不呈严格线性，只要排序一致，该得分就会很高。

# _cosine (余弦相似度)：计算两个向量夹角的余弦值，只关注方向，不关注长度
{'pearson_cosine': 0.3710938716460552,
 'spearman_cosine': 0.45148122260403883,

# _manhattan (曼哈顿距离)：计算向量在各坐标轴上的绝对轴距总和
 'pearson_manhattan': 0.4037396904694362,
 'spearman_manhattan': 0.4396893995197567,

# _euclidean (欧几里得距离)：计算两点之间的直线距离（物理距离）
 'pearson_euclidean': 0.390788259199341,
 'spearman_euclidean': 0.43444104358464286,

# _dot (点积/内积)：同时结合了向量的方向与长度（未做归一化）
 'pearson_dot': 0.3392927926047231,
 'spearman_dot': 0.3530708415227247,

# 分数最高的（一般来说，优秀的 BERT 微调模型在 STS-B 上的 Spearman 系数通常能达到 0.8 以上）
 'pearson_max': 0.4037396904694362,
 'spearman_max': 0.45148122260403883}


我们最感兴趣的是 pearson_cosine
pearson_cosine 的值为 0.3710938716460552，作为基准值
"""
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.3710938716460552,
 'spearman_cosine': 0.45148122260403883,
 'pearson_manhattan': 0.4037396904694362,
 'spearman_manhattan': 0.4396893995197567,
 'pearson_euclidean': 0.390788259199341,
 'spearman_euclidean': 0.43444104358464286,
 'pearson_dot': 0.3392927926047231,
 'spearman_dot': 0.3530708415227247,
 'pearson_max': 0.4037396904694362,
 'spearman_max': 0.45148122260403883}

# MTEB（海量文本嵌入基准测试）

In [ ]:
from mteb import MTEB

"""
为了统一评估过程，大规模文本嵌入基准（Massive Text Embedding Benchmark，MTEB）应运而生。
MTEB 涵盖 8 个嵌入任务，涉及 58 个数据集和 112 种语言
"""

# 选择评估任务
# Choose evaluation task
evaluation = MTEB(tasks=["Banking77Classification"])

# 计算结果
# Calculate results
results = evaluation.run(embedding_model)

"""
这个JSON展示的是你的文本嵌入（Embedding）模型在 MTEB（Massive Text Embedding Benchmark，海量文本嵌入基准） 中的一个 多分类任务（Classification） 评测结果

- Banking77 是一个专门针对银行/金融客服场景的意图分类数据集
- 测试逻辑：模型需要将输入的银行客户提问（文本），准确分类到这 77 个类别中的某一个。因为类别极多且语义相近，该任务非常考验模型对金融专业文本的微观语义分辨能力。

说明：
目前主流的优秀开源 Embedding 模型（如 BGE、cohere-embed、stella 等）的 Accuracy/F1 分数普遍在 0.80 ~ 0.85（80%~85%） 以上

{'Banking77Classification': {'mteb_version': '1.1.2',
  'dataset_revision': '0fd18e25b25c072e09e0d92ab615fda904d66300',
  'mteb_dataset_name': 'Banking77Classification',
  'test': {'accuracy': 0.46022727272727276,     # accuracy (准确率)。测试集里，模型每判定 100 个客户的意图，只有约 46 个分类是完全正确的
   'f1': 0.45802738001849663,                   # 综合考虑了精确率（Precision）和召回率（Recall）的调和平均数。说明模型在 77 个不同类别上的表现相对均衡，没有出现严重的“偏科”
   'accuracy_stderr': 0.009556987908238961,     # 标准误差
   'f1_stderr': 0.01072225943077292,            # 标准误差

   'main_score': 0.46022727272727276,           # 分数最高
   'evaluation_time': 29.63}}}                  # 评估耗时
"""
results

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- Banking77Classification, s2s

/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid = os.fork()


{'Banking77Classification': {'mteb_version': '1.1.2',
  'dataset_revision': '0fd18e25b25c072e09e0d92ab615fda904d66300',
  'mteb_dataset_name': 'Banking77Classification',
  'test': {'accuracy': 0.46022727272727276,
   'f1': 0.45802738001849663,
   'accuracy_stderr': 0.009556987908238961,
   'f1_stderr': 0.01072225943077292,
   'main_score': 0.46022727272727276,
   'evaluation_time': 29.63}}}

⚠️ **VRAM Clean-up** - You will need to run the code below to partially empty the VRAM (GPU RAM). If that does not work, it is advised to restart the notebook instead. You can check the resources on the right-hand side (if you are using Google Colab) to check whether the used VRAM is indeed low. You can also run `!nivia-smi` to check current usage.

In [ ]:
# # Empty and delete trainer/model
# trainer.accelerator.clear()
# del trainer, embedding_model

# # Garbage collection and empty cache
# import gc
# import torch

# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

# Loss Fuctions（损失函数）

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

## Cosine Similarity Loss(余弦相似度)

In [ ]:
from datasets import Dataset, load_dataset

# 从GLUE加载MNLI数据集
# 0 = 蕴含, 1 = 中性, 2 = 矛盾
# Load MNLI dataset from GLUE
# 0 = entailment, 1 = neutral, 2 = contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

# (neutral/contradiction)=0 and (entailment)=1
mapping = {2: 0, 1: 0, 0: 1}
train_dataset = Dataset.from_dict({
    "sentence1": train_dataset["premise"],
    "sentence2": train_dataset["hypothesis"],
    "label": [float(mapping[label]) for label in train_dataset["label"]]
})

In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# 创建评估器
# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score / 5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [ ]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# 余弦相似度损失函数
# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="cosineloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Step,Training Loss
100,0.231900
200,0.168900
300,0.170900
400,0.157800
500,0.152900
600,0.156100
700,0.149300
800,0.154500
900,0.150900
1000,0.145600


Computing widget examples:   0%|          | 0/5 [00:00<?, ?example/s]

Computing widget examples:   0%|          | 0/5 [00:00<?, ?example/s]

Computing widget examples:   0%|          | 0/5 [00:00<?, ?example/s]

TrainOutput(global_step=1563, training_loss=0.15676780793427353, metrics={'train_runtime': 364.4779, 'train_samples_per_second': 137.183, 'train_steps_per_second': 4.288, 'total_flos': 0.0, 'train_loss': 0.15676780793427353, 'epoch': 1.0})

In [ ]:

"""
pearson_cosine 的值为 0.72，与使用 softmax 损失函数的示例（pearson_cosine 的值为 0.59）
相比有了很大的提升。这显示了损失函数对模型性能的影响

{'pearson_cosine': 0.7222320710908391,
 'spearman_cosine': 0.725059765496038,
 'pearson_manhattan': 0.7338172618636865,
 'spearman_manhattan': 0.7323465534428775,
 'pearson_euclidean': 0.7332726423686017,
 'spearman_euclidean': 0.7316943270141215,
 'pearson_dot': 0.6603672299249149,
 'spearman_dot': 0.6624301208511642,
 'pearson_max': 0.7338172618636865,
 'spearman_max': 0.7323465534428775}
"""
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.7222320710908391,
 'spearman_cosine': 0.725059765496038,
 'pearson_manhattan': 0.7338172618636865,
 'spearman_manhattan': 0.7323465534428775,
 'pearson_euclidean': 0.7332726423686017,
 'spearman_euclidean': 0.7316943270141215,
 'pearson_dot': 0.6603672299249149,
 'spearman_dot': 0.6624301208511642,
 'pearson_max': 0.7338172618636865,
 'spearman_max': 0.7323465534428775}

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## Multiple Negatives Ranking Loss(多负例排序损失函数)

In [ ]:
import random
from tqdm import tqdm
from datasets import Dataset, load_dataset

# MNLI（Multi-Genre Natural Language Inference）多领域自然语言推理
# Load MNLI dataset from GLUE
mnli = load_dataset("glue", "mnli", split="train").select(range(50_000))  # 只取 5 0000条数据
mnli = mnli.remove_columns("idx")

# 0 = entailment, 1 = neutral, 2 = contradiction
# 标签0 代表 Entailment（蕴含）, 这意味着 premise（前提）和 hypothesis（假设）在语义上是完全成立、互为正确因果或同义的
# 通过这个过滤，代码确保了留下的每一条数据里，premise 和 hypothesis 是一对天然的、绝对正确的“正例对（Positive Pair）
mnli = mnli.filter(lambda x: True if x['label'] == 0 else False)  # 筛选出蕴含关系的文字作为 锚点-正例

# 构建三元组数据集
# Prepare data and add a soft negative
train_dataset = {"anchor": [], "positive": [], "negative": []}

# 构建软负例
soft_negatives = mnli["hypothesis"]
random.shuffle(soft_negatives)  # 随机打乱，作为如软负例
for row, soft_negative in tqdm(zip(mnli, soft_negatives)):
    train_dataset["anchor"].append(row["premise"])  # 锚点，比如：如何用 Python画图
    train_dataset["positive"].append(row["hypothesis"])  # 正例：matplotlib绘图教程
    train_dataset["negative"].append(soft_negative)  # 反例：今天晚饭吃什么
train_dataset = Dataset.from_dict(train_dataset)

# 我们只选择了标注为 entailment 的句子, 因此行数从 50 000 减少到了 16 875
len(train_dataset)

16875it [00:01, 14110.96it/s]


16875

In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# 创建评估器
# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score / 5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [ ]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# 多负例排序损失函数
# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="mnrloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Step,Training Loss
100,0.345200
200,0.105500
300,0.079000
400,0.062200
500,0.069000


Computing widget examples:   0%|          | 0/5 [00:00<?, ?example/s]

TrainOutput(global_step=528, training_loss=0.12795479415041028, metrics={'train_runtime': 180.866, 'train_samples_per_second': 93.301, 'train_steps_per_second': 2.919, 'total_flos': 0.0, 'train_loss': 0.12795479415041028, 'epoch': 1.0})

In [ ]:
"""
与我们之前使用余弦相似度损失函数训练的模型（pearson_cosine 的值为 0.72）相比，
使用 MNR 损失函数训练的模型（pearson_cosine 的值为 0.80）似乎更加准确

{'pearson_cosine': 0.8070727434643791,
 'spearman_cosine': 0.8106193672462586,
 'pearson_manhattan': 0.8213132116968124,
 'spearman_manhattan': 0.8164551132664518,
 'pearson_euclidean': 0.820988086354926,
 'spearman_euclidean': 0.8160139830687847,
 'pearson_dot': 0.7429357515240518,
 'spearman_dot': 0.7316164586329814,
 'pearson_max': 0.8213132116968124,
 'spearman_max': 0.8164551132664518}
"""
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.8070727434643791,
 'spearman_cosine': 0.8106193672462586,
 'pearson_manhattan': 0.8213132116968124,
 'spearman_manhattan': 0.8164551132664518,
 'pearson_euclidean': 0.820988086354926,
 'spearman_euclidean': 0.8160139830687847,
 'pearson_dot': 0.7429357515240518,
 'spearman_dot': 0.7316164586329814,
 'pearson_max': 0.8213132116968124,
 'spearman_max': 0.8164551132664518}

# **Fine-tuning（微调）**

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## **Supervised（监督）**

In [ ]:
from datasets import load_dataset
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load MNLI dataset from GLUE
# 0 = entailment, 1 = neutral, 2 = contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score / 5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [ ]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# 我们可以使用预训练的嵌入模型sentence-transformers来替代 bert-base-uncased：
# Define model
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="finetuned_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

Step,Training Loss
100,0.155500
200,0.110000
300,0.118600
400,0.115300
500,0.110700
600,0.101000
700,0.113100
800,0.099800
900,0.109600
1000,0.105800


Computing widget examples:   0%|          | 0/5 [00:00<?, ?example/s]

Computing widget examples:   0%|          | 0/5 [00:00<?, ?example/s]

Computing widget examples:   0%|          | 0/5 [00:00<?, ?example/s]

TrainOutput(global_step=1563, training_loss=0.10982195932897176, metrics={'train_runtime': 117.3739, 'train_samples_per_second': 425.989, 'train_steps_per_second': 13.316, 'total_flos': 0.0, 'train_loss': 0.10982195932897176, 'epoch': 1.0})

In [ ]:
"""
0.85 的分数是我们目前看到的最高分，但别忘了我们用于微调的预训练模型已经在完整的
MNLI 数据集上进行了训练，而我们只使用了 50 000 个样本

{'pearson_cosine': 0.8489503881223601,
 'spearman_cosine': 0.8484667083117318,
 'pearson_manhattan': 0.8503843871673679,
 'spearman_manhattan': 0.8475679105384369,
 'pearson_euclidean': 0.8513072191805562,
 'spearman_euclidean': 0.8484667083117318,
 'pearson_dot': 0.8489503890256918,
 'spearman_dot': 0.8484667083117318,
 'pearson_max': 0.8513072191805562,
 'spearman_max': 0.8484667083117318}
"""
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.8489503881223601,
 'spearman_cosine': 0.8484667083117318,
 'pearson_manhattan': 0.8503843871673679,
 'spearman_manhattan': 0.8475679105384369,
 'pearson_euclidean': 0.8513072191805562,
 'spearman_euclidean': 0.8484667083117318,
 'pearson_dot': 0.8489503890256918,
 'spearman_dot': 0.8484667083117318,
 'pearson_max': 0.8513072191805562,
 'spearman_max': 0.8484667083117318}

In [ ]:
# Evaluate the pre-trained model
original_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
evaluator(original_model)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'pearson_cosine': 0.8696194608752055,
 'spearman_cosine': 0.8671637433378804,
 'pearson_manhattan': 0.8670399009851635,
 'spearman_manhattan': 0.8663946139224048,
 'pearson_euclidean': 0.867871599362501,
 'spearman_euclidean': 0.8671643653432983,
 'pearson_dot': 0.8696194616795601,
 'spearman_dot': 0.8671631197908374,
 'pearson_max': 0.8696194616795601,
 'spearman_max': 0.8671643653432983}

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## **Augmented SBERT（增强的Sentence双向编码器表示）**

**Step 1:** Fine-tune a cross-encoder，

使用小型标注数据集（黄金数据集）微调交叉编码器（BERT）

In [ ]:
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from sentence_transformers import InputExample
from sentence_transformers.datasets import NoDuplicatesDataLoader

# 在原始的 50 000 个文档中选取 10 000 个文档组成小数据集，以模拟只有有限标注数据的情况
# Prepare a small set of 10000 documents for the cross-encoder
dataset = load_dataset("glue", "mnli", split="train").select(range(10_000))
mapping = {2: 0, 1: 0, 0: 1}

# 准备黄金数据集
# Data Loader
gold_examples = [
    # 使用了数据集自带的 row["label"]（即 人类专家标注的真实标签 Ground Truth，代码中通过 mapping 将其转换为了 0 和 1 的二分类标签）
    InputExample(texts=[row["premise"], row["hypothesis"]], label=mapping[row["label"]])
    for row in tqdm(dataset)
]
gold_dataloader = NoDuplicatesDataLoader(gold_examples, batch_size=32)

# 使用pandas DataFrame，以更方便地处理数据
# Pandas DataFrame for easier data handling
gold = pd.DataFrame(
    {
        'sentence1': dataset['premise'],
        'sentence2': dataset['hypothesis'],
        'label': [mapping[label] for label in dataset['label']]  # 是有标注的，且代表了我们的真实标注
    }
)

100%|██████████| 10000/10000 [00:00<00:00, 25870.92it/s]


In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder

# 在黄金数据集上训练交叉编码器
# Train a cross-encoder on the gold dataset
cross_encoder = CrossEncoder('bert-base-uncased', num_labels=2)
# 微调
cross_encoder.fit(
    train_dataloader=gold_dataloader,  # 黄金数据集
    epochs=1,
    show_progress_bar=True,
    warmup_steps=100,
    use_amp=False
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/312 [00:00<?, ?it/s]

**Step 2:** Create new sentence pairs，

创建新的句子对

In [ ]:
# 使用剩余的 40 000 个句子对（来自包含 50 000 个句子对的原始数据集）作为白银数据集
# Prepare the silver dataset by predicting labels with the cross-encoder
silver = load_dataset("glue", "mnli", split="train").select(range(10_000, 50_000))
pairs = list(zip(silver['premise'], silver['hypothesis']))  # 它只有 premise 和 hypothesis 文本，而没有（或不使用）人类的真实标签

**Step 3:** Label new sentence pairs with the fine-tuned cross-encoder (silver dataset),

使用微调后的交叉编码器标注新的句子对（白银数据集）

In [ ]:
import numpy as np

# 通过微调的交叉编码器 来 标注（预测/推理）这些 句子对
# Label the sentence pairs using our fine-tuned cross-encoder
output = cross_encoder.predict(pairs, apply_softmax=True, show_progress_bar=True)
silver = pd.DataFrame(
    {
        "sentence1": silver["premise"],
        "sentence2": silver["hypothesis"],
        "label": np.argmax(output, axis=1)
    }
)

**Step 4:** Train a bi-encoder (SBERT) on the extended dataset (gold + silver dataset)，

在扩展数据集（黄金数据集 + 白银数据集）上训练双编码器（SBERT）

In [ ]:
# 组合数据集
# Combine gold + silver
data = pd.concat([gold, silver], ignore_index=True, axis=0)
data = data.drop_sduplicates(subset=['sentence1', 'sentence2'], keep="first")
train_dataset = Dataset.from_pandas(data, preserve_index=False)

In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score / 5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [ ]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="augmented_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,  # 增强后的数据集
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

In [ ]:
"""
余弦相似度损失函数示例的原始版本在使用完整数据集时得分为 0.72，
而我们仅使用其中 1/5 = 20% 的数据，就获得了 0.71 的分数！

{'pearson_cosine': 0.7101597020018693,
 'spearman_cosine': 0.7210536464320728,
 'pearson_manhattan': 0.7296749443525249,
 'spearman_manhattan': 0.7284184255293913,
 'pearson_euclidean': 0.7293097297208753,
 'spearman_euclidean': 0.7282830906742256,
 'pearson_dot': 0.6746605824703588,
 'spearman_dot': 0.6754486790570754,
 'pearson_max': 0.7296749443525249,
 'spearman_max': 0.7284184255293913}
"""
# Evaluate our trained model
evaluator(embedding_model)

In [ ]:
trainer.accelerator.clear()

**Step 5**: Evaluate without silver dataset，

只评估黄金数据集

In [ ]:
# Combine gold
data = pd.concat([gold], ignore_index=True, axis=0)
data = data.drop_duplicates(subset=['sentence1', 'sentence2'], keep="first")
train_dataset = Dataset.from_pandas(data, preserve_index=False)

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="gold_only_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

In [ ]:
# Evaluate our trained model
evaluator(embedding_model)

Compared to using both the silver and gold datasets, using only the gold dataset reduces the performance of the model!

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## **Unsupervised Learning（无监督学习）**

### Tranformer-based Denoising AutoEncoder (TSDAE)

In [ ]:
"""
我们只需要一组不带任何标注的句子，因此训练这个模型非常简单直接。
首先，下载一个外部分词器，用于去噪过程

Natural Language Toolkit（自然语言工具箱）
"""
# Download additional tokenizer
import nltk

nltk.download('punkt')

In [ ]:
from tqdm import tqdm
from datasets import Dataset, load_dataset
from sentence_transformers.datasets import DenoisingAutoEncoderDataset

"""
从数据中创建普通的句子，并移除所有标签以模拟无监督的设置
"""

# 创建一个一维的句子列表
# Create a flat list of sentences
mnli = load_dataset("glue", "mnli", split="train").select(range(25_000))
flat_sentences = mnli["premise"] + mnli["hypothesis"]

# 为输入数据添加噪声
# Add noise to our input data
damaged_data = DenoisingAutoEncoderDataset(list(set(flat_sentences)))

# 创建数据集
# Create dataset
train_dataset = {"damaged_sentence": [], "original_sentence": []}
for data in tqdm(damaged_data):
    train_dataset["damaged_sentence"].append(data.texts[0])
    train_dataset["original_sentence"].append(data.texts[1])
train_dataset = Dataset.from_dict(train_dataset)

In [ ]:
"""
第一个句子展示了带噪声的数据，而第二个句子展示了原始句子。

{'damaged_sentence': 'Grim jaws are.',
 'original_sentence': 'Grim faces and hardened jaws are not people-friendly.'}
"""
train_dataset[0]

In [ ]:
# # Choose a different deletion ratio
# flat_sentences = list(set(flat_sentences))
# damaged_data = DenoisingAutoEncoderDataset(
#     flat_sentences,
#     noise_fn=lambda s: DenoisingAutoEncoderDataset.delete(s, del_ratio=0.6)
# )

In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score / 5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [1]:
from sentence_transformers import models, SentenceTransformer

# Create your embedding model
word_embedding_model = models.Transformer('bert-base-uncased')
"""
使用 [CLS] 词元作为池化策略，而不是对词元嵌入进行平均池化。
在关于 TSDAE 的论文中，这被证明更有效，因为平均池化会丢失位置信息，而使用 [CLS] 词元则不会

池化策略：
1、CLS 策略：
原理：BERT 在处理文本时，会在每句话的最前面强制加上一个特殊符号 [CLS]。BERT 内部的自注意力机制会把整句话的精髓都聚集在这个 [CLS] 位置上。
做法：池化层直接抛弃后面所有字词的向量，只摘取第一个 Token（即 [CLS]）的向量作为整句话的向量。

2、MEAN 策略（最常用、效果通常最好）
原理：取平均值。
做法：把这句话里所有字词的向量加起来，然后除以句子的长度。把所有字词的贡献平均一下，得到代表整句话的向量。
在 SentenceTransformer 官方的标准模型中，默认大多使用 MEAN 池化。

3、MAX 策略
原理：取最大值。
做法：在 768 个维度的每一个维度上，都挑出所有字词中数值最大的那一个，拼成一个新的向量。意在捕捉句中最强烈的特征。

示例：
【输入文本】 -> "我 喜欢 机器学习"
                     │
【BERT 层】   -> 输出了 6 个字向量: [V我, V喜, V欢, V机, V器, V学] (每个 768 维)
                     │
【池化层】   -> 执行 'cls' 策略（直接抽取首位 `[CLS]` 向量）
                     │
【最终输出】 -> 得到 1 个唯一的句向量 (768 维) -> 此时可以用来算余弦相似度了！

"""
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), 'cls')
embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

KeyboardInterrupt: 

In [ ]:
from sentence_transformers import losses

"""
使用我们的句子对，我们需要一个损失函数来尝试使用噪声句子重构原始句子，这个损失函数即 DenoisingAutoEncoderLoss（去噪自编码器损失函数）
我们绑定了两个模型的参数: 编码器的嵌入层和解码器的输出层不使用单独的权重，而是共享权重。这意味着一个层的权重更新也会反映在另一个层中
"""
# 使用去噪自编码器损失函数
# Use the denoising auto-encoder loss
train_loss = losses.DenoisingAutoEncoderLoss(
    embedding_model, tie_encoder_decoder=True
)
train_loss.decoder = train_loss.decoder.to("cuda")

In [ ]:
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="tsdae_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=16, # 减少批量大小，这个损失函数会增加内存使用量
    per_device_eval_batch_size=16,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

In [ ]:
"""
{'pearson_cosine': 0.6991809700971775,
 'spearman_cosine': 0.713693213167873,
 'pearson_manhattan': 0.7152343356643568,
 'spearman_manhattan': 0.7201441944880915,
 'pearson_euclidean': 0.7151142243297436,
 'spearman_euclidean': 0.7202291660769805,
 'pearson_dot': 0.5198066451871277,
 'spearman_dot': 0.5104025515225046,
 'pearson_max': 0.7152343356643568,
 'spearman_max': 0.7202291660769805}
"""
# Evaluate our trained model
evaluator(embedding_model)

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()